Imports:

In [ ]:
import torch
import torch.nn.functional as F
from torch.distributions import Normal, kl_divergence

torch.manual_seed(0)

 Part 1 - Closed Form KL Divergence

In [ ]:

def kl_divergence_gaussian(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    """
    Closed-form KL divergence between N(mu, exp(logvar)) and N(0, 1).
    Returns a scalar (summed over all dimensions and batch).
    """
    # YOUR CODE HERE
    kl = -0.5 * torch.sum(
        1 + logvar - mu**2 - torch.exp(logvar)
    )

    return kl

In [ ]:
mu     = torch.randn(32, 16)   # batch of 32, latent dim 16
logvar = torch.randn(32, 16)
sigma  = torch.exp(0.5 * logvar)

your_kl = kl_divergence_gaussian(mu, logvar)

q = Normal(mu, sigma)
p = Normal(torch.zeros_like(mu), torch.ones_like(sigma))
lib_kl  = kl_divergence(q, p).sum()

print(f"Your KL:   {your_kl:.4f}")
print(f"Torch KL:  {lib_kl:.4f}")
assert torch.isclose(your_kl, lib_kl, atol=1e-4), "KL values don't match!"
print("✅ Part 1 passed")

Your KL:   460.9495
Torch KL:  460.9495
✅ Part 1 passed



 Part 2 - Reparameterization Trick


In [ ]:
def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
    """
    Sample z using the reparameterization trick.
    z = mu + std * eps, where eps ~ N(0, I)
    """
    # YOUR CODE HERE
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    z = mu + std * eps

    return z

In [ ]:
mu_test     = torch.tensor([2.0, -1.0])
logvar_test = torch.tensor([0.0,  0.5])

samples = torch.stack([reparameterize(mu_test, logvar_test) for _ in range(10000)])
print(f"Target mu:       {mu_test.tolist()}")
print(f"Sample mean:     {samples.mean(0).tolist()}")
print(f"Target std:      {torch.exp(0.5 * logvar_test).tolist()}")
print(f"Sample std:      {samples.std(0).tolist()}")
# Means and stds should be close (within ~0.05)
print("✅ Part 2 passed")

Target mu:       [2.0, -1.0]
Sample mean:     [2.017857789993286, -1.0104761123657227]
Target std:      [1.0, 1.2840254306793213]
Sample std:      [1.0110328197479248, 1.2827435731887817]
✅ Part 2 passed


 Part 3 - ELBO Loss

In [ ]:
def elbo_loss(x: torch.Tensor,
              x_recon: torch.Tensor,
              mu: torch.Tensor,
              logvar: torch.Tensor) -> torch.Tensor:
    """
    ELBO loss = Reconstruction loss + KL divergence
    - Reconstruction: BCE between x_recon and x, summed over pixels and batch
    - KL: closed-form KL between N(mu, exp(logvar)) and N(0, I)
    Returns a scalar.
    """
    # YOUR CODE HERE
    recon_loss = F.binary_cross_entropy(
        x_recon,
        x,
        reduction='sum'
    )

    kl_loss = kl_divergence_gaussian(
        mu,
        logvar
    )

    return recon_loss + kl_loss

In [ ]:
batch, dim, latent = 16, 784, 32
x      = torch.rand(batch, dim)
x_recon = torch.sigmoid(torch.randn(batch, dim))
mu     = torch.randn(batch, latent)
logvar = torch.randn(batch, latent)

loss = elbo_loss(x, x_recon, mu, logvar)
print(f"ELBO loss (should be a positive scalar): {loss.item():.4f}")
assert loss.ndim == 0, "Loss must be a scalar!"
print("✅ Part 3 passed")

ELBO loss (should be a positive scalar): 10541.8613
✅ Part 3 passed


# Part 4 - Reflection

### 1. Why can't we maximize log p(x) directly?

because to compute log p(x) we need to integrate over latent variables z which is not feasable (intactable). The encoder q(z|x) allows us to optimize the ELBO instead.

---

### 2. What happens if we remove the KL term?

it makes the latent variable space messy and disorganized, generating wrong images with garbage z

---

### 3. Why does the reparameterization trick work?

The reparameterization trick rewrites the sampling operation as

z = mu + sigma * eps

where eps ~ N(0,1). This separates randomness from the learnable parameters, allowing gradients to flow through mu and sigma.

---

### 4. What happens when mu=0 and logvar=0?

variance becomes 1 and  q(z|x)=N(0,1), which exactly matches the prior distribution resulting in zero KL Divergence